In [420]:
import time
import random
import json
from pydantic import BaseModel
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors
from unittest.mock import MagicMock
from dataclasses import dataclass, field

In [421]:
load_dotenv()
client = genai.Client()

### Gemini api call

In [422]:
@dataclass
class API_Configs:
    model_config = types.GenerateContentConfig(
    system_instruction="Respond STRICTLY with JSON format. Do not add markdown fences.",
    max_output_tokens=8192,
)
    
    model_tiers = {
    "high": {"model_name": "gemini-3.5-flash"},
    "medium": {"model_name": "gemini-2.5-flash"},
    "low": {"model_name": "gemini-3.1-flash-lite"}
}

In [423]:
@dataclass
class Retry_Config:
    max_attempts: int = 3
    base_delay: float = 1.0
    max_delay: float = 30.0

    retry_status_codes = (429, 500, 503, 504)

In [424]:
tier = API_Configs.model_tiers.get(model_priority.lower(), API_Configs.model_tiers["medium"])
model_name = tier["model_name"]

In [425]:
def execute_prompt(model_name, prompt):
    for attempt in range (1, Retry_Config.max_attempts + 1):

        # Api call Section
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print(f"Sending initial request...")

            stream = client.models.generate_content_stream(model = model_name, contents = prompt, config = API_Configs.model_config)
            
            #Consumes the streamed chunks during execution so that errors can be caught early instead of waiting for the output to be parsed.
            for chunk in stream:
                yield chunk

            return 
        
        # Error handling section
        except errors.APIError as e:

            # Refers to the Tuple of status codes that can be retried in Retry_Config.
            if e.code in Retry_Config.retry_status_codes:
                if attempt == Retry_Config.max_attempts:
                    print("Max attempts reached.")
                    raise e

                # Exponential backoff, i should probably add a way to check the server header for the exact time needed if it exists later (keyword: *later*)
                new_delay = random.uniform(0.5, 1.5) * Retry_Config.base_delay * (2 ** (attempt - 1))
                clamped_delay = min(new_delay, Retry_Config.max_delay)

                print(f" Code {e.code}. Backing off. Waiting {clamped_delay:.2f}s...")
                time.sleep(clamped_delay)
                continue

            else:
                status_reason = getattr(e, "status", "UNKNOWN")
                
                if e.code == 404 or status_reason == "NOT_FOUND": # Where is this error? i cant find it? Muhehehehehehhe
                    print(f"Model Not Found [404]: '{model_name}' does not exist or is deprecated.")
                    
                elif e.code in (401, 403) or status_reason == "PERMISSION_DENIED":
                    print(f"Authentication Failed [{e.code}]: Check your API key, project quotas, or permissions.")
                    
                elif e.code == 400 or status_reason == "INVALID_ARGUMENT":
                    print(f"Invalid Argument [400]: Malformed structure or prompt content limits exceeded.")
                    
                else:
                    print(f"Non-retryable error [{status_reason}] ({e.code}): {e.message}")
                    
                raise e

In [426]:
def get_structured_output(model_name, prompt):
    response_stream = execute_prompt(model_name = model_name, prompt = prompt, )

    full_response = "".join(getattr(chunk, "text", "") or "" for chunk in response_stream)

    data = json.loads(full_response)
    return data

In [427]:
model_priority = "high"

In [434]:
#prompt = "You shall henceforth and at once provide me with TEN of the greatest most glorified spaghettie recipes that have crossed your existence."
prompt = "Say hello in 10 random languages and state which language it is"

In [435]:
result = get_structured_output(model_name = model_name, prompt = prompt)


Sending initial request...


In [436]:
print(json.dumps(result, indent=4))

{
    "greetings": [
        {
            "language": "Spanish",
            "greeting": "Hola"
        },
        {
            "language": "French",
            "greeting": "Bonjour"
        },
        {
            "language": "German",
            "greeting": "Hallo"
        },
        {
            "language": "Japanese",
            "greeting": "Konnichiwa"
        },
        {
            "language": "Mandarin Chinese",
            "greeting": "N\u01d0 h\u01ceo"
        },
        {
            "language": "Russian",
            "greeting": "Zdravstvuyte"
        },
        {
            "language": "Arabic",
            "greeting": "Marhaba"
        },
        {
            "language": "Hindi",
            "greeting": "Namaste"
        },
        {
            "language": "Portuguese",
            "greeting": "Ol\u00e1"
        },
        {
            "language": "Korean",
            "greeting": "Annyeonghaseyo"
        }
    ]
}
